# Task 1  
EDA đơn biến, EDA đa biến, đề xuất các cột cần loại bỏ trong quá trình huấn luyện và các cột cần xử lý null, outlier

* Các cột đề xuất loại bỏ  
`event_type` (Purchase Chunk), `is_deleted` (User, Purchase, Item Chunks), `date_key` (Purchase Chunk), `event_value` (Purchase Chunk), `item_id` (Item Chunk), `manufacturer` (Item Chunk), `brand` (Item Chunk), `gender_target` (Item Chunk)
* Xử lý Outlier (Ngoại lệ) / Chuẩn hóa / Biến đổi  
`user_id` (Purchase Chunk), `item_id` (Purchase Chunk), `event_value` (Purchase Chunk), `price` (Purchase Chunk, Item Chunk), `quantity` (Purchase Chunk), `discount` (Purchase Chunk), `weight` (Item Chunk), `timestamp` (Purchase Chunk)
* Xử lý Mất cân bằng (Imbalance) / Gộp nhóm (Grouping)  
`gender` (User Chunk), `location`, `province`, `region` (User Chunk), `membership` (User Chunk), `install_app` (User Chunk), `payment` (Purchase Chunk), `category_l1` (Item Chunk), `category_l2` (Item Chunk), `category_l3` (Item Chunk)

# Task 2
* Loại bỏ cột theo đề xuất từ task 1 và dựa vào độ tương đồng giữa các cột
* Xử lí các giá trị null và outlier
* chuẩn hóa định dạng dữ liệu trong các cột
* Tạo đặc trưng mới

### User Chunk
* Các cột được giữ lại:
`customer_id`, `user_id`, `gender`, `location`, `province`, `membership`, `install_app`
* Các cột bị loại bỏ:
    * sync_status_id: Bị loại bỏ vì gần như toàn bộ giá trị là Null, phần còn lại không có giá trị phân biệt (đều bằng 2).
    * location_name: Dữ liệu quá chi tiết, rời rạc và thiếu tính khái quát.
    * is_deleted: Tất cả giá trị đều là False, không mang thông tin phân biệt.
* Các cột được chuẩn hóa:
    * province: Chuẩn hóa bằng cách loại bỏ các tiền tố "Thành Phố " và "Tỉnh " để thống nhất các giá trị.
## Purchase Chunk
* Các cột được giữ lại: `timestamp`, `user_id`, `item_id`, `price`, `quantity`, `customer_id`, `created_date`, `payment`, `location`, `discount`, `channel`     
* Các cột được chuẩn hóa:
    * Cột price: Áp dụng kỹ thuật Winsorizing.
    * Cột discount: Chuẩn hóa thành tỷ lệ phần trăm bằng cách lấy discount chia cho price.
## Item Chunk
* Các cột được giữ lại: created_date, creation_timestamp, p_id, price, gp, category, category_l1, gender_target, sale_status
* Các cột được chuẩn hóa:
    * price và gp (Gross Profit): Áp dụng kỹ thuật Winsorizing với công thức 1.5 * IQR để xử lý các outlier.
    * gender_target: Xử lý giá trị Null (Fill NA) bằng cách dựa vào thông tin từ các cột khác bằng cách tìm kiếm các từ khóa (ví dụ: 'bé gái', 'bé trai', 'sơ sinh', 'unisex') trong các cột description, description_new, category, category_l1 để suy luận và điền vào các giá trị còn thiếu.
## Tạo mới đặc trưng
* Tạo cột mới object_target
* Xây dựng một từ điển hoàn chỉnh phân loại sản phẩm theo hai nhóm chính:
    * Theo Đối tượng sử dụng: "Bé Gái", "Bé Trai", "Sơ sinh", "Unisex", "Nữ" (cho mẹ bầu & sau sinh), "Nam", "Người lớn" (dùng cho gia đình).
    * Theo Nhu cầu/Mục đích: "Dinh dưỡng & Cho ăn" (sữa, đồ ăn dặm, yếm), "Sức khỏe & An toàn" (vitamin, tã, kem chống hăm), "Đồ chơi & Phát triển" (gặm nướu, sách, xe tập đi).
* Tìm kiếm từ khóa của các category trong 6 cột description, description_new, category, category_l1, category_l2, category_l3 và fill các giá trị 'không xác định' bằng các category


# Task 2.1

## Task 2.1A 
* tìm kiếm và đếm các cặp sản phẩm thường xuyên cùng nhau xuất hiện trong 1 lần mua hàng
* tạo mới cột top10_bought_together trong item chunk lưu 10 item_id của 10 sản phẩm thường được mua cùng với sản phẩm này nhất
* tạo mới 2 file CSV df_cooc_info.csv và cooc_info.csv để lưu trữ chi tiết tất cả các cặp sản phẩm mua chung và thông tin (brand, category) của chúng
## Task 2.1B 
* Ước tính tuổi của em bé bằng cách phân tích lịch sử mua hàng (df_purchase) và thông tin sản phẩm (df_item). 
    * tìm ngày đầu tiên (timestamp) mà một khách hàng mua các sản phẩm đặc trưng cho trẻ sơ sinh (như sữa "step 1", tã "0M", hoặc "bình sữa") để suy luận ra tuổi của bé
* tạo ra một DataFrame mới (df_result) bằng cách thêm 6 cột đặc trưng mới:
    * first_day_step_1 (Ngày đầu tiên mua sữa "step 1")
    * age_by_step1 (Tuổi của bé dựa trên ngày mua "step 1")
    * first_day_age (Ngày đầu tiên mua đồ sơ sinh (0M, 0-3M))
    * age_by_age (Tuổi của bé dựa trên ngày mua đồ sơ sinh)
    * first_day_milk (Ngày đầu tiên mua đồ dùng liên quan đến sữa)
    * age_by_milk (Tuổi của bé dựa trên ngày mua đồ dùng sữa)